# 🌍 Notebook 3: Consistent Hashing in Practice

## What You'll Learn

1. Build a **Redis router** that uses consistent hashing to distribute keys across 3 Redis nodes
2. See what happens when a node goes **down** and comes back **up**
3. How **Redis Cluster** uses fixed hash slots (CRC16 % 16384) instead of a ring
4. How **DynamoDB** and **Cassandra** use consistent hashing with virtual nodes

> **Prerequisites:**
> - Complete Notebooks 1 and 2
> - Run `docker compose up -d` to start the 3 Redis nodes

In [ ]:
import redis
import hashlib
import bisect
import time
from collections import Counter, defaultdict

---

## Setup: Connecting to Our Redis Nodes

We have 3 Redis instances running in Docker on ports 6380, 6381, and 6382.
Think of these as 3 cache servers in a distributed system.

```
┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│  Redis       │   │  Redis       │   │  Redis       │
│  Node 1     │   │  Node 2     │   │  Node 3     │
│  port 6380  │   │  port 6381  │   │  port 6382  │
└─────────────┘   └─────────────┘   └─────────────┘
```

Our Python code will decide which node each key goes to — that's the consistent hashing part!

In [ ]:
# Connect to all 3 Redis nodes
REDIS_NODES = {
    "redis-node-1": {"host": "localhost", "port": 6380, "decode_responses": True},
    "redis-node-2": {"host": "localhost", "port": 6381, "decode_responses": True},
    "redis-node-3": {"host": "localhost", "port": 6382, "decode_responses": True},
}

clients = {}
for name, config in REDIS_NODES.items():
    try:
        client = redis.Redis(**config)
        client.ping()
        clients[name] = client
        print(f"  ✅ Connected to {name} (port {config['port']})")
    except redis.ConnectionError:
        print(f"  ❌ Cannot connect to {name} (port {config['port']})")
        print(f"     Run: docker compose up -d")

print(f"\n📡 {len(clients)} Redis nodes ready")

In [ ]:
# Flush all nodes to start clean
for name, client in clients.items():
    client.flushdb()
print("🧹 All nodes flushed clean")

---

## Building a Consistent Hash Router for Redis

This class wraps our consistent hash ring with actual Redis connections.
It decides **which Redis node** to use for each key, then executes the command on that node.

In [ ]:
class RedisHashRouter:
    """
    Routes Redis commands to the correct node using consistent hashing.
    This is similar to what a Redis Cluster client does internally.
    """

    def __init__(self, num_virtual_nodes: int = 150):
        self.num_virtual_nodes = num_virtual_nodes
        self.ring = {}              # ring position -> node name
        self.sorted_positions = []
        self.clients = {}           # node name -> redis.Redis client
        self.stats = {"reads": 0, "writes": 0, "errors": 0}

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

    def add_node(self, name: str, client: redis.Redis):
        """Add a Redis node to the router."""
        self.clients[name] = client
        for i in range(self.num_virtual_nodes):
            pos = self._hash(f"{name}#vn{i}")
            self.ring[pos] = name
            bisect.insort(self.sorted_positions, pos)

    def remove_node(self, name: str):
        """Remove a Redis node (e.g., when it goes down)."""
        if name in self.clients:
            del self.clients[name]
        for i in range(self.num_virtual_nodes):
            pos = self._hash(f"{name}#vn{i}")
            if pos in self.ring:
                del self.ring[pos]
                self.sorted_positions.remove(pos)

    def _get_node_name(self, key: str) -> str:
        """Find which node owns this key."""
        if not self.ring:
            return None
        pos = self._hash(key)
        idx = bisect.bisect_right(self.sorted_positions, pos)
        if idx == len(self.sorted_positions):
            idx = 0
        return self.ring[self.sorted_positions[idx]]

    def set(self, key: str, value: str, ttl: int = None) -> bool:
        """SET a key on the correct Redis node."""
        node_name = self._get_node_name(key)
        try:
            client = self.clients[node_name]
            if ttl:
                client.setex(key, ttl, value)
            else:
                client.set(key, value)
            self.stats["writes"] += 1
            return True
        except Exception as e:
            self.stats["errors"] += 1
            return False

    def get(self, key: str) -> str:
        """GET a key from the correct Redis node."""
        node_name = self._get_node_name(key)
        try:
            client = self.clients[node_name]
            self.stats["reads"] += 1
            return client.get(key)
        except Exception as e:
            self.stats["errors"] += 1
            return None

    def get_key_location(self, key: str) -> str:
        """Show which node a key would be routed to (for debugging)."""
        return self._get_node_name(key)

print("✅ RedisHashRouter class ready!")

---

## Distributing Data Across Redis Nodes

Let's store 1,000 user profiles across our 3 Redis nodes using the router.

In [ ]:
# Create the router and add all 3 nodes
router = RedisHashRouter(num_virtual_nodes=150)
for name, client in clients.items():
    router.add_node(name, client)

# Store 1,000 user profiles
for i in range(1000):
    key = f"user:{i}"
    value = f'{{"id": {i}, "name": "User {i}", "email": "user{i}@example.com"}}'
    router.set(key, value)

print(f"✅ Stored 1,000 user profiles\n")

# Check distribution across nodes
print("📊 Keys per Redis node:")
node_counts = Counter()
for i in range(1000):
    node = router.get_key_location(f"user:{i}")
    node_counts[node] += 1

for node, count in sorted(node_counts.items()):
    bar = "█" * (count // 10)
    print(f"  {node}: {count:>4} keys  {bar}")

print(f"\n💡 Keys are distributed roughly evenly across all 3 nodes.")

In [ ]:
# Verify we can read back all keys from the correct nodes
found = 0
missing = 0
for i in range(1000):
    value = router.get(f"user:{i}")
    if value:
        found += 1
    else:
        missing += 1

print(f"🔍 Read-back verification:")
print(f"  Found:   {found:,} / 1,000 ✅")
print(f"  Missing: {missing:,} / 1,000")
print(f"\n  Stats: {router.stats}")

---

## Simulating Node Failure

What happens when one of our Redis nodes goes down?
In a real system, this could be a server crash, network partition, or maintenance.

Let's simulate losing `redis-node-2` and see the impact.

In [ ]:
# Record where each key lives BEFORE the failure
before_failure = {}
for i in range(1000):
    before_failure[f"user:{i}"] = router.get_key_location(f"user:{i}")

# Count keys on the doomed node
keys_on_node2 = sum(1 for v in before_failure.values() if v == "redis-node-2")
print(f"⚠️  redis-node-2 currently holds {keys_on_node2} keys")
print(f"   Simulating failure...\n")

# Remove node 2 from the router (simulating it going down)
router.remove_node("redis-node-2")

# Check where keys are routed now
after_failure = {}
for i in range(1000):
    after_failure[f"user:{i}"] = router.get_key_location(f"user:{i}")

# How many keys changed their target node?
keys_moved = sum(1 for k in before_failure if before_failure[k] != after_failure[k])
print(f"📊 Impact of losing redis-node-2:")
print(f"   Keys that changed node: {keys_moved} / 1,000 ({keys_moved/10:.1f}%)")
print(f"   Keys that stayed put:   {1000 - keys_moved} / 1,000\n")

# Where did node-2's keys go?
moved_to = Counter()
for k in before_failure:
    if before_failure[k] == "redis-node-2":
        moved_to[after_failure[k]] += 1

print(f"   redis-node-2's {keys_on_node2} keys were redistributed to:")
for node, count in sorted(moved_to.items()):
    print(f"     → {node}: {count} keys")
print(f"\n   ✅ Load is spread evenly across remaining nodes!")

In [ ]:
# But those keys are LOST (the data was on node-2's Redis instance)
# Let's see how many keys we can still read
readable = 0
lost = 0
for i in range(1000):
    value = router.get(f"user:{i}")
    if value:
        readable += 1
    else:
        lost += 1

print(f"🔍 After node-2 failure:")
print(f"  Keys still readable: {readable:,} (on nodes 1 and 3) ✅")
print(f"  Keys lost:           {lost:,} (were on node 2) ❌")
print(f"\n💡 In production, you'd use REPLICATION to prevent data loss.")
print(f"   Each key would be stored on 2-3 nodes, so losing one node")
print(f"   doesn't lose any data. The hash ring just tells you the")
print(f"   PRIMARY node; replicas go on the next N-1 nodes clockwise.")

### Recovering: Adding the Node Back

In [ ]:
# Simulate node-2 coming back online
node2_client = redis.Redis(host="localhost", port=6381, decode_responses=True)
node2_client.ping()
router.add_node("redis-node-2", node2_client)

# Check new distribution
after_recovery = Counter()
for i in range(1000):
    node = router.get_key_location(f"user:{i}")
    after_recovery[node] += 1

print(f"✅ redis-node-2 is back!\n")
print(f"📊 Key distribution after recovery:")
for node, count in sorted(after_recovery.items()):
    bar = "█" * (count // 10)
    print(f"  {node}: {count:>4} keys  {bar}")
print(f"\n💡 In production, node-2 would need to fetch its keys from")
print(f"   replicas or a backup to rebuild its data.")

---

## How Redis Cluster Actually Works

Redis Cluster does **not** use a consistent hash ring. Instead, it uses a simpler
approach with **16,384 fixed hash slots**:

```
  slot = CRC16(key) % 16384
```

Each node is assigned a **range** of slots:

```
┌───────────────────────────────────────────────┐
│ Slot 0        5460  5461       10922  10923   16383 │
│ │─── Node A ───│  │─── Node B ───│  │─── Node C ─│ │
└───────────────────────────────────────────────┘
```

**Why not a hash ring?** Redis chose simplicity:
- Fixed slot count makes rebalancing predictable
- Slot-to-node mapping stored in a simple table
- Moving slots between nodes is explicit (not implicit like ring rebalancing)

Let's simulate Redis Cluster's approach:

In [ ]:
def crc16(data: bytes) -> int:
    """CRC16-CCITT used by Redis Cluster."""
    crc = 0
    for byte in data:
        crc ^= byte << 8
        for _ in range(8):
            if crc & 0x8000:
                crc = (crc << 1) ^ 0x1021
            else:
                crc = crc << 1
            crc &= 0xFFFF
    return crc

def redis_cluster_slot(key: str) -> int:
    """Calculate the Redis Cluster hash slot for a key."""
    return crc16(key.encode()) % 16384

# Demo: see which slots some keys map to
print("Redis Cluster hash slot assignment:\n")
slot_ranges = {
    "Node A": (0, 5460),
    "Node B": (5461, 10922),
    "Node C": (10923, 16383),
}

for key in ["user:1", "user:2", "user:100", "order:42", "product:999"]:
    slot = redis_cluster_slot(key)
    node = next(n for n, (lo, hi) in slot_ranges.items() if lo <= slot <= hi)
    print(f"  '{key}' \u2192 slot {slot:>5} \u2192 {node}")

In [ ]:
# Distribution of 10,000 keys across Redis Cluster slots
slot_distribution = Counter()
for i in range(10_000):
    slot = redis_cluster_slot(f"key:{i}")
    node = next(n for n, (lo, hi) in slot_ranges.items() if lo <= slot <= hi)
    slot_distribution[node] += 1

print("📊 Redis Cluster: 10,000 keys across 3 nodes:\n")
for node, count in sorted(slot_distribution.items()):
    bar = "█" * (count // 50)
    print(f"  {node}: {count:>5,} keys  {bar}")
print(f"\n✅ Very even! Fixed slot ranges guarantee balanced distribution.")

---

## How DynamoDB Uses Consistent Hashing

Amazon's DynamoDB uses consistent hashing with virtual nodes, but adds several
sophisticated features:

```
            DynamoDB Architecture
  ┌───────────────────────────────────────┐
  │                                       │
  │   Partition Key (e.g., user_id)        │
  │          │                              │
  │          ▼                              │
  │   Consistent Hash → Partition           │
  │          │                              │
  │          ▼                              │
  │   Replicate to 3 Availability Zones    │
  │     (AZ-1, AZ-2, AZ-3)                 │
  │          │                              │
  │          ▼                              │
  │   Leader election via Paxos/Raft        │
  │                                       │
  └───────────────────────────────────────┘
```

**Key DynamoDB features:**

| Feature | How It Works |
|---------|-------------|
| **Partition key** | Your table's primary key is hashed to determine the partition |
| **Virtual nodes** | Each storage node owns multiple token ranges on the ring |
| **3-way replication** | Each partition is replicated across 3 Availability Zones |
| **Automatic splitting** | When a partition gets too large, DynamoDB splits it automatically |
| **Adaptive capacity** | Hot partitions can borrow throughput from cooler ones |

> **Interview tip:** When asked about DynamoDB partitioning, mention that it uses
> consistent hashing with virtual nodes, 3-way replication, and automatic partition splitting.

---

## How Cassandra Uses Consistent Hashing

Apache Cassandra is perhaps the most faithful implementation of consistent hashing
in production:

```
        Cassandra Token Ring

             Token 0
          ╭─────────╮
        ╯   Node A    ╰
      ╯    (vnodes:    ╰
     │    256 tokens)   │
     │                  │    Each node owns 256 token
   Node D            Node B   ranges (virtual nodes)
     │                  │    scattered around the ring.
      ╰               ╯
        ╰   Node C   ╯
          ╰─────────╯
          Token 2^63
```

**Key Cassandra features:**

| Feature | How It Works |
|---------|-------------|
| **Murmur3 partitioner** | Uses MurmurHash3 (faster than MD5) for token assignment |
| **vnodes** (default: 256) | Each node owns 256 random positions on the ring |
| **Replication factor (RF)** | Data copied to RF consecutive nodes clockwise on the ring |
| **Tunable consistency** | Choose per-query: ONE, QUORUM, ALL reads/writes |
| **Gossip protocol** | Nodes discover ring membership through peer-to-peer gossip |

> **Interview tip:** Cassandra's vnodes (virtual nodes) are exactly the concept from
> Notebook 1. The default of 256 vnodes per server provides excellent balance.

---

## 📊 Comparison: Hashing Strategies in Production

| Feature | Redis Cluster | DynamoDB | Cassandra |
|---------|--------------|----------|----------|
| **Hashing** | CRC16 % 16384 | Consistent hash ring | Consistent hash ring (Murmur3) |
| **Slots/Tokens** | 16,384 fixed slots | Virtual nodes | 256 vnodes per node (default) |
| **Rebalancing** | Manual slot migration | Automatic | Automatic with streaming |
| **Replication** | Async replica per master | 3 AZs (synchronous) | Configurable RF (1-N) |
| **Consistency** | Strong (single master) | Strong or eventual | Tunable (ONE to ALL) |
| **Node discovery** | Cluster bus (gossip) | Managed by AWS | Gossip protocol |
| **When to use** | Cache, sessions | Managed NoSQL, serverless | Write-heavy, multi-DC |

---

## 🤔 When Should You Care About Consistent Hashing?

### You DON'T need to implement it when:
- Using **managed databases** (DynamoDB, Cloud Spanner) — they handle it for you
- Using **Redis Cluster** — the client library handles slot routing
- Using **Cassandra** — the driver handles token-aware routing

### You DO need to understand it when:
- **Designing** a distributed cache, database, or message broker from scratch
- **Interview questions** about distributed systems
- **Debugging** why data isn't where you expect in a distributed system
- Building **custom load balancers** or **CDN routing**

### Key points for interviews:
1. **Modulo hashing** is bad because adding/removing servers moves ~75% of keys
2. **Consistent hashing** with virtual nodes moves only ~1/N keys
3. **Virtual nodes** ensure even load distribution
4. **Replication** (not hashing) handles fault tolerance
5. **Redis Cluster** uses fixed hash slots, not a ring — a valid design trade-off

---

## 🧹 Cleanup

In [ ]:
# Flush all Redis nodes
for name, client_cfg in REDIS_NODES.items():
    try:
        c = redis.Redis(**client_cfg)
        c.flushdb()
    except Exception:
        pass
print("✅ All Redis nodes cleaned up!")
print("\n💡 To stop the Docker containers: docker compose down")

## 🎯 Key Takeaways

1. Consistent hashing is the **foundation** of data distribution in systems like DynamoDB and Cassandra
2. **Redis Cluster** chose a simpler fixed-slot approach — both are valid designs
3. In practice, you rarely implement consistent hashing yourself — but understanding it helps you:
   - Choose the right partition key
   - Debug data distribution issues
   - Ace system design interviews
4. **Replication** works alongside hashing to provide fault tolerance
5. The choice between consistent hashing and fixed slots is a real **design trade-off**:
   - Hash ring = flexible, automatic rebalancing
   - Fixed slots = simpler, more predictable

## 📚 Further Reading

- [DynamoDB Paper](https://www.allthingsdistributed.com/files/amazon-dynamo-sosp2007.pdf) — the original Dynamo paper that popularized consistent hashing
- [Cassandra Architecture](https://cassandra.apache.org/doc/latest/cassandra/architecture/) — token ring and vnodes
- [Redis Cluster Specification](https://redis.io/docs/reference/cluster-spec/) — hash slots and CRC16